In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# The Cost of Attention

The language model of that section was trained on
contexts of 128 bytes. Language models in use today read whole reports,
code repositories and books, often tens or hundreds of thousands of tokens
at once. Self-attention lets every token read every other token in a
single layer, and the cost is one score for every pair of positions. How
does this cost grow with the length of the context, and how can it be
reduced?

Doubling the context quadruples the number of pairs. A context of 8,192
tokens already has 67 million query--key pairs per head, and storing their
scores in single precision takes 256 MiB for one head of one sequence.
Arithmetic and memory both grow with the square of the sequence length, and
at long contexts this quadratic term dominates the cost of the whole layer.

We first compare self-attention with the convolutional and recurrent layers of
earlier chapters, count its cost, and check the count against measurements on
a GPU. The rest of the section develops three ways to reduce that cost. The
first changes only the order of the computation. It computes exact attention
one block of keys at a time, without ever storing the full matrix, and
FlashAttention fits this schedule to the memory hierarchy of a GPU. The
second scores fewer pairs: each token attends only to a window of recent
tokens, and depth restores the long-range paths. The third replaces the
softmax by a similarity kernel that factorizes, which turns attention into a
recurrent network whose state is a matrix. We end by timing the three side
by side.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Quadratic Time and Memory

### CNNs, RNNs, and Self-Attention

Consider the problem of mapping a sequence of $n$ tokens, each a
$d$-dimensional vector, to another sequence of the same shape. Three familiar
architectures can perform this mapping: a one-dimensional CNN
(that section), an RNN (that section), and self-attention
(that section). the figure
draws all three as graphs over the sequence. Two properties of these graphs
matter beyond raw arithmetic. The number of *sequential* operations bounds how
much of the work can run in parallel on modern hardware. The *maximum path
length* counts how many layer transformations connect two positions. Path
length is a property of the architecture. A short path avoids multiplying many
recurrent Jacobians, but it does not by itself guarantee that training finds a
long-range dependency.

![Sequence-layer connectivity for five tokens. Highlighted edges show how the inputs reach the third output in one layer: a convolution with kernel size $k = 3$ sees its neighbors, a recurrence passes information step by step along the chain, and self-attention connects every position directly. Below each panel are the path length between distant tokens and whether the outputs are computed in parallel.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-cnn-rnn-self-attention.svg)

A convolutional layer with kernel size $k$ and $d$ channels in and out costs
$O(knd^2)$ operations: each of the $n$ positions mixes $k$
neighbors through a $d \times d$ map. All positions are computed at once, so
there are $O(1)$ sequential steps, but the receptive field grows by
only $k-1$ positions per layer. Connecting a pair of tokens $n$ apart takes a
stack of depth $O(n/k)$. An RNN requires $O(nd^2)$ for the
sequence, one $d \times d$ state update per token, but the updates are
ordered. It takes $O(n)$ sequential steps, and a signal from the
first token reaches the last only through $O(n)$ applications of
the cell.

Self-attention first projects the inputs at cost
$O(nd^2)$, then computes an $n \times d$ by $d \times n$ product and
an $n \times n$ by $n \times d$ product at cost $O(n^2d)$. These
matrix multiplications have $O(1)$ sequential depth, and any token
can attend to any other in one layer, giving path length $O(1)$.
the table collects the accounting.

<!-- tbl-caption:Per-layer cost of mapping $n$ tokens of dimension $d$ to $n$ tokens of dimension $d$. -->

| layer type | complexity | sequential operations | maximum path length |
|:--|:--|:--|:--|
| convolution (kernel $k$) | $O(knd^2)$ | $O(1)$ | $O(n/k)$ |
| recurrence | $O(nd^2)$ | $O(n)$ | $O(n)$ |
| self-attention | $O(nd^2+n^2d)$ | $O(1)$ | $O(1)$ |

Self-attention has constant sequential depth and path length, but its mixing
term is quadratic in $n$. The table includes the query, key, value, and
output projections so that all three rows count the complete layer. When
$n$ and $d$ are comparable, the linear projection and quadratic mixing terms
have similar order. For $n\gg d$, the mixing term dominates. The remainder
of this section studies ways to reduce its memory use or arithmetic cost.

### Counting FLOPs

The leading constants make this comparison more precise. One attention layer
receives $n$ queries, keys, and values of dimension $d$ and computes
$\mathrm{softmax}(\mathbf{Q}\mathbf{K}^\top/\sqrt{d})\mathbf{V}$
(the equation). The score matrix $\mathbf{Q}\mathbf{K}^\top$
costs $2n^2 d$ floating-point operations, because each of its $n^2$ entries
is a length-$d$ dot product and we count one multiply--add as two FLOPs.
The softmax costs a few operations per entry, $O(n^2)$, and the
value mixing $\mathrm{softmax}(\cdot)\mathbf{V}$ another $2n^2 d$. In
total

$$
\underbrace{4 n^2 d}_{\textrm{scores and mixing}} + \; O(n^2)
\quad \textrm{FLOPs}.
$$

The four projections around attention in a multi-head layer cost $8nd^2$,
which is linear in $n$, and the head count drops out of both terms
(the equation). Setting $4n^2d = 8nd^2$ locates the
crossover: the quadratic part dominates the layer as soon as $n > 2d$. GPT-2
small, with $d = 768$ and contexts of 1,024 tokens, sits below the
crossover, so its projections still cost more than its scores. A model with
$d = 4096$ at $n = 131{,}072$ sits far beyond it. There $n/2d = 16$, so an
attention layer does sixteen times as much score-and-mix work as projection
work.

### Counting Memory

Arithmetic is only one part of the cost, and at long contexts memory is
often the binding constraint. A naive implementation materializes the score
matrix and its softmax. These are two $n \times n$ buffers of activations, $8n^2$
bytes in single precision, per head and per sequence in the batch. During
training the attention weights are also saved for the backward pass. The
projections, by contrast, keep only $O(nd)$ activations. At
$n = 8192$ and a head dimension of $d_h = 64$, the setting of the
experiments below, the inputs $\mathbf{Q}, \mathbf{K}, \mathbf{V}$
occupy 6 MiB together, while the two score buffers occupy 512 MiB. At a
context of $n = 131{,}072$ tokens a *single* attention map in fp32 would
take $n^2 \times 4$ bytes $\approx 69$ GB (per head, per sequence), which
is why no system at that scale ever stores one.

We check the formula against a measurement on one head. The function
`attention` computes $\mathrm{softmax}(\mathbf{Q}\mathbf{K}^\top/\sqrt{d})
\mathbf{V}$ through the full score matrix, and `d2l.peak_memory` of
that section reports the temporary memory that a call needs. The
prediction is the two $n \times n$ float buffers, $8n^2$ bytes.

Measuring per-op memory at runtime is awkward in JAX, for two reasons. XLA
*preallocates* a large fraction of GPU memory as an arena at startup, so
device-side counters describe the arena, not the operation. The compiler may
also fuse or rematerialize operations, so the buffers it materializes are
determined during compilation rather than directly by the Python source. The
reliable answer therefore comes from the compiler itself. Lowering a jitted
function and compiling it yields a `memory_analysis()` report of how much
temporary buffer space the executable reserves.

In [ ]:
def attention(Q, K, V):
    """softmax(Q K^T / sqrt(d)) V through the full score matrix."""
    scores = Q @ K.swapaxes(-2, -1) / math.sqrt(Q.shape[-1])
    return jax.nn.softmax(scores, axis=-1) @ V

We measure a single head of width 64 as $n$ doubles.

In [ ]:
d_h = 64
for n in [2048, 4096, 8192, 16384]:
    Q = jax.random.normal(d2l.get_key(), (n, d_h))
    mem = d2l.peak_memory(attention, Q, Q, Q)
    print(f'n = {n:5d}: XLA temp {mem/2**20:7.1f}'
          f' MiB, predicted 8n^2 B = {8*n*n/2**20:7.1f} MiB')

The measurement matches $8n^2$ bytes up to a small extra workspace, and
doubling $n$ quadruples it. At $n = 16{,}384$, a single fp32 head already
needs 2 GiB of temporary storage.

The running time has the same quadratic dependence. We time the forward pass
as $n$ doubles, after warm-up calls. On an accelerator, kernel launches
return before the work is done, so we synchronize before reading the clock.
`d2l.Benchmark` of that section does both and reports the average
time per call.

In [ ]:
attention_jit = jax.jit(attention)
for n in [2048, 4096, 8192, 16384]:
    Q = jax.random.normal(d2l.get_key(), (n, d_h))
    print(d2l.Benchmark(lambda: attention_jit(Q, Q, Q), desc=f'n = {n:5d}'))

At small $n$, the time does not yet follow the operation count: the GPU is
not saturated, launch overhead weighs in, and the smallest score matrices
still fit in the GPU's on-chip cache. Beyond a few thousand tokens, each
doubling of $n$ multiplies the time by about four, the growth of the $4n^2d$
operation count.

## Exact Attention Without the Matrix

### Blockwise Online Softmax

The full $n \times n$ matrix need not be stored. The apparent obstacle to
computing attention piecewise is the softmax. Each weight
$\exp(a_j)/\sum_{j'}\exp(a_{j'})$ depends on *all* scores of its row
through the normalizer and, for numerical safety, through the row maximum
subtracted before exponentiation. Both quantities can be maintained
*online*, the way one computes a running mean
[@Milakov.Gimelshein.2018]. Process the keys in blocks and keep, per
query, three running statistics: the maximum $m$ seen so far, the sum $s$ of
exponentials rescaled to that maximum, and the output accumulator
$\mathbf{o}$. A new block of scores $\{a_j\}$ with values
$\{\mathbf{v}_j\}$ updates them as

$$
\begin{aligned}
m' &= \max\big(m, \max_j a_j\big), \\
s' &= s\, e^{m - m'} + \sum\nolimits_j e^{a_j - m'}, \\
\mathbf{o}' &= \mathbf{o}\, e^{m - m'} + \sum\nolimits_j e^{a_j - m'}\, \mathbf{v}_j,
\end{aligned}
$$

and after the last block, $\mathbf{o}/s$ is exactly
$\mathrm{softmax}(\mathbf{a})\mathbf{V}$. Whenever a new block raises the
maximum, the factor $e^{m - m'}$ rescales all quantities accumulated under
the old maximum, so no approximation is made.
the figure shows the resulting schedule. One
$n \times c$ stripe of the score matrix is in memory at a time, next to a
running state of size $O(nd)$ [@Rabe.Staats.2021].

![Online softmax processes the score matrix one key block at a time. Of the scores, only the current $n \times c$ stripe is in memory, next to the running statistics $\mathbf{m}, \mathbf{s}, \mathbf{O}$; the full $n \times n$ matrix is never materialized.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-online-softmax.svg)

### A Chunked Implementation

From here on we work in the causal setting of a language model. Excluded
scores are overwritten with the most negative finite value of the dtype, and
`d2l.causal_mask` supplies the mask, as in
that section. We first implement a naive
reference, which accepts any number of leading axes such as heads, and then a
chunked version for one head. The chunked implementation applies
the equation to all queries at once and processes one key block
per iteration.

In [ ]:
def causal_attention(Q, K, V):
    """Reference: causal attention with the full score matrix."""
    scores = Q @ K.swapaxes(-2, -1) / math.sqrt(Q.shape[-1])
    mask = d2l.causal_mask(Q.shape[-2])
    scores = jnp.where(mask, scores, jnp.finfo(scores.dtype).min)
    return jax.nn.softmax(scores, axis=-1) @ V

def chunked_attention(Q, K, V, chunk_size=512):
    """Exact causal attention, one n-by-chunk_size block at a time."""
    (n, d), d_v = Q.shape, V.shape[-1]
    assert n % chunk_size == 0, 'lax.scan needs equal-sized blocks'
    pos_q = jnp.arange(n)[:, None]
    def block(carry, chunk):
        m, s, O = carry
        Kc, Vc, start = chunk
        pos_k = start + jnp.arange(chunk_size)
        scores = Q @ Kc.T / math.sqrt(d)
        scores = jnp.where(pos_k[None, :] > pos_q,
                           jnp.finfo(scores.dtype).min, scores)
        m_new = jnp.maximum(m, scores.max(axis=-1, keepdims=True))
        scale = jnp.exp(m - m_new)                    # Rescale the past
        P = jnp.exp(scores - m_new)
        s = s * scale + P.sum(axis=-1, keepdims=True)
        O = O * scale + P @ Vc
        return (m_new, s, O), None
    init = (jnp.full((n, 1), jnp.finfo(Q.dtype).min), jnp.zeros((n, 1)),
            jnp.zeros((n, d_v)))
    chunks = (K.reshape(-1, chunk_size, d), V.reshape(-1, chunk_size, d_v),
              jnp.arange(0, n, chunk_size))           # Blocks, their starts
    (m, s, O), _ = jax.lax.scan(block, init, chunks)
    return O / s

The chunked version is meant to be exact, so we check it against the
reference.

In [ ]:
n, d_h = 2048, 64
Q, K, V = (jax.random.normal(k, (n, d_h))
           for k in jax.random.split(d2l.get_key(), 3))
# Full fp32 matrix products: the GPU default (TF32) would perturb the two
# computations differently, at the 1e-4 level
with jax.default_matmul_precision('highest'):
    err = jnp.abs(chunked_attention(Q, K, V)
                  - causal_attention(Q, K, V)).max()
print(f'maximum deviation: {float(err):.2e}')

The two computations agree up to floating-point rounding, at the level of
$10^{-7}$ to $10^{-6}$ in fp32. Chunking changes the order of evaluation,
not the definition of attention. The chunked version holds $n \times c$
scores at a time instead of $n \times n$, so its footprint should grow
linearly in $n$ rather than quadratically.

In [ ]:
lengths, mems = [2048, 4096, 8192, 16384], [[], []]
for n in lengths:
    Q = jax.random.normal(d2l.get_key(), (n, d_h))
    mems[0].append(d2l.peak_memory(causal_attention, Q, Q, Q) / 2**20)
    mems[1].append(d2l.peak_memory(chunked_attention, Q, Q, Q) / 2**20)
d2l.plot(lengths, mems, 'sequence length n', 'XLA temp memory (MiB)',
         legend=['full matrix', 'chunked'], xscale='log', yscale='log')

At $n = 16{,}384$ the full-matrix implementation needs about 2 GiB of
scratch memory. The chunked one needs more than an order of magnitude less,
and the gap doubles with every further doubling of $n$. The result and the
$4n^2d$ FLOP count are unchanged. Only the summation order and the memory
schedule differ.

### Memory Traffic as the Computational Bottleneck

Chunking also reduces data movement. On a modern GPU, memory bandwidth often
limits attention more than arithmetic throughput. An accelerator can execute
hundreds of arithmetic operations in the time it takes to fetch one float
from off-chip memory. Naive attention writes the $n^2$ score matrix to slow
memory and reads it back for the softmax and again for the value mixing. The
chunked schedule keeps each stripe in fast on-chip memory, finishes all work
on it, and never writes it out. FlashAttention [@Dao.Fu.Ermon.ea.2022]
is this algorithm fitted to the hardware, with tile sizes matched to on-chip
SRAM and softmax statistics kept in registers. During training, the backward
pass recomputes the stripes instead of storing the weights, because the extra
arithmetic takes less time than the memory traffic it avoids. The result is
exact attention that is faster *and* needs asymptotically less memory.

Later versions keep the algorithm and change its schedule. FlashAttention-2
divides by the normalizer only once at the end, as `chunked_attention` does,
and cuts each sequence into blocks of queries that run in parallel, so that a
single long sequence still keeps the whole GPU busy [@Dao.2024].
FlashAttention-3 overlaps data movement, matrix products, and exponentials
on newer GPUs and adds an 8-bit floating-point mode
[@Shah.Bikshandi.Zhang.ea.2024]. In 16-bit precision every version
computes exact attention; only the schedule follows the hardware. Frameworks
provide it as a fused kernel:
`torch.nn.functional.scaled_dot_product_attention` in PyTorch and
`jax.nn.dot_product_attention` in JAX. We compare it with the
naive reference on a minibatch of 2 sequences with 8 heads of dimension 64
in half precision at $n = 8192$.

In [ ]:
def fused_attention(Q, K, V):
    """Causal attention in the fused cuDNN kernel."""
    # The kernel takes (batch, length, heads, width): swap axes 1 and 2
    Q, K, V = (A.swapaxes(1, 2) for A in (Q, K, V))
    O = jax.nn.dot_product_attention(Q, K, V, is_causal=True,
                                     implementation='cudnn')
    return O.swapaxes(1, 2)

B, H, n = 2, 8, 8192
X = jax.random.normal(d2l.get_key(), (B, H, n, d_h), dtype=jnp.float16)
for name, f in [('naive', causal_attention), ('fused', fused_attention)]:
    f_jit = jax.jit(f)
    t = d2l.Benchmark(lambda: f_jit(X, X, X)).time
    mem = d2l.peak_memory(f, X, X, X)
    print(f'{name}: {t*1e3:6.2f} ms, XLA temp {mem/2**20:7.1f} MiB')

The fused kernel is several times faster than the naive implementation, and
it needs almost no temporary memory where the naive one needs gigabytes.
Both gains come from keeping the score blocks out of off-chip memory. In
practice, dense attention on sequences beyond a few hundred tokens therefore
runs in a fused kernel.

`implementation='cudnn'` selects cuDNN's fused attention, which accepts half
and bfloat16 inputs on NVIDIA GPUs. The default implementation is XLA's own,
which materializes the score matrix.

## Windowed and Sparse Attention

### Sliding-Window Attention

FlashAttention removes the $n^2$ *memory*, but every query still scores
every key, so the arithmetic remains quadratic. To reduce it too, we must
decide that some query--key pairs are not worth scoring. The simplest
useful rule is locality: each query attends only to the $w$ most recent
positions. With the masks of that section, the
causal mask becomes a band: causal, and at most $w - 1$ positions back.

In [ ]:
def window_mask(n, w):
    """Causal band: query i may attend to keys i - w + 1, ..., i."""
    i = jnp.arange(n)
    return d2l.causal_mask(n) & (i[:, None] - i[None, :] < w)

With a window of 4, each row of random scores keeps its weight on the
query's own position and the three before it:

In [ ]:
scores = jax.random.normal(d2l.get_key(), (12, 12))
d2l.show_heatmaps(d2l.masked_softmax(scores, window_mask(12, 4))[None, None],
                  xlabel='Keys', ylabel='Queries')

### Receptive Field Growth with Depth

A window of width $w$ excludes direct interactions beyond $w$ tokens, but
depth increases the receptive field as in CNNs (that section). A query
in layer 2 attends to keys that are themselves layer-1 outputs, each of
which already summarizes its own window. Information therefore hops up to
$w-1$ positions per layer, and after $L$ layers a token's receptive field
spans $1 + L(w-1)$ positions. We can compute this directly. The band mask,
read as an adjacency matrix, composes across layers by boolean matrix
multiplication, and its $L$-th power says which positions can influence
which through $L$ layers.

In [ ]:
n, w, depths = 64, 8, [1, 2, 4]
band = window_mask(n, w).astype(jnp.float32)
reach, maps = jnp.eye(n), []
for L in range(1, max(depths) + 1):
    reach = ((reach @ band) > 0).astype(jnp.float32)
    if L in depths:
        maps.append(reach)
        print(f'depth {L}: last query reaches {int(reach[-1].sum())} '
              f'positions (formula: {min(n, 1 + L * (w - 1))})')
d2l.show_heatmaps(jnp.stack(maps)[None], xlabel='Influencing position',
                  ylabel='Position', titles=[f'depth {L}' for L in depths],
                  figsize=(8, 3.2))

The count matches the formula, and the heatmaps show the band widening layer
by layer. Mistral 7B uses this growth: it attends through a window of 4096 in
each of its 32 layers, for a reach beyond a hundred thousand tokens
[@Jiang.Sablayrolles.Mensch.ea.2023]. the table
describes what is given up. A stack of windows has path length
$O(n/w)$ instead of $O(1)$, so with $w$ in the thousands
it assumes that few dependencies need more than a small number of hops.

More recent models therefore interleave windowed layers with a few layers of
full attention, called *global* layers. Gemma 3 follows every five windowed
layers of width 1024 with one global layer [@Gemma.Team.2025], and
gpt-oss alternates global layers with windows of 128 tokens
[@OpenAI.2025]. The global layers restore the $O(1)$ path
length. The windowed layers keep both the arithmetic and the memory of
generation small: a model that generates text caches the keys and values of
past tokens (that section), and a windowed layer needs
to keep only the last $w$ of them. With $r$ windowed layers per global
layer, the cache shrinks to

$$
\frac{r \min(n, w) + n}{(r + 1)\, n}
$$

of the cache of a stack of global layers only: each of the $r$ windowed
layers caches $\min(n, w)$ positions and the global layer all $n$, against
$(r+1)n$ positions for global layers throughout. For Gemma 3's $r = 5$ and
$w = 1024$, this is about a quarter at $n = 8192$, and it approaches $1/6$
as the context grows.

Llama 4 also drops the positional encoding from its global layers, which
then work like the NoPE of that section. It keeps
the rotary encoding only in its local layers, which attend within chunks of
the sequence and therefore see only short offsets [@Meta.2025]. The aim
is a model that runs at contexts longer than those it was trained on.

### A Linear-Cost Implementation

The mask defines the result, but by itself it saves no work. A masked
softmax over a band still scores all $n^2$ pairs and then discards most of
them. The efficient implementation processes the sequence
in blocks of $w$ queries. A query can attend only to keys in its own block
and in the block before it, so each block scores a $w \times 2w$ tile. The
total work is $2nw$ scores instead of $n^2$, which is linear in $n$ for fixed
$w$.

In [ ]:
def windowed_attention(Q, K, V, w):
    """Causal sliding-window attention in O(nw) time and memory."""
    n, d = Q.shape
    assert n % w == 0, 'blocks must tile the sequence exactly'
    Qb, Kb, Vb = (X.reshape(n // w, w, -1) for X in (Q, K, V))
    # Each query block attends to its own key block and the one before it
    prev = lambda X: jnp.concatenate([jnp.zeros_like(X[:1]), X[:-1]])
    Kb = jnp.concatenate([prev(Kb), Kb], 1)
    Vb = jnp.concatenate([prev(Vb), Vb], 1)
    i = jnp.arange(w)[:, None]                         # Query in its block
    j = jnp.arange(-w, w)[None, :]                     # Key offset from it
    first = jnp.arange(n // w)[:, None, None] == 0
    mask = (j <= i) & (i - j < w) & ~(first & (j < 0))  # No keys before 0
    scores = Qb @ Kb.swapaxes(-1, -2) / math.sqrt(d)
    return (d2l.masked_softmax(scores, mask) @ Vb).reshape(n, -1)

The blocked version should agree with the mask-based definition.

In [ ]:
n, w = 2048, 128
Q, K, V = (jax.random.normal(k, (n, d_h))
           for k in jax.random.split(d2l.get_key(), 3))
with jax.default_matmul_precision('highest'):
    scores = Q @ K.T / math.sqrt(d_h)
    reference = d2l.masked_softmax(scores, window_mask(n, w)) @ V
    err = jnp.abs(windowed_attention(Q, K, V, w) - reference).max()
print(f'maximum deviation: {float(err):.2e}')

A fused kernel cannot skip single scores. It computes in tiles of
$b \times b$ scores held in on-chip memory, and it can skip a tile only if
the mask excludes all of it. Tiles inside the mask are computed without
masking, and only tiles that straddle its edge apply the mask entry by
entry. The cost of a mask in a fused kernel is therefore the fraction of
tiles it touches. We count it for tiles of $b = 128$, the usual size.

In [ ]:
def tiles_visited(mask, b):
    """Fraction of b-by-b tiles of a mask that hold at least one True."""
    T = mask.shape[0] // b
    return float(mask.reshape(T, b, T, b).any(axis=(1, 3)).mean())

n, b, w = 2048, 128, 128
print(f'causal:  {tiles_visited(d2l.causal_mask(n), b):.1%} of tiles')
print(f'window:  {tiles_visited(window_mask(n, w), b):.1%} of tiles')

A causal mask touches just over half of the tiles, and a window of 128
touches about one in eight: the diagonal tile and its left neighbor in each
row of tiles. A windowed fused kernel therefore costs time proportional to
$nw$, rounded up to whole tiles, and narrowing the window below the tile
size saves nothing more. Fused kernels take the window as part of their
interface, and we check that their output equals that of
`windowed_attention`:

In [ ]:
Q, K, V = (jax.random.normal(k, (1, n, 1, d_h))
           for k in jax.random.split(d2l.get_key(), 3))
with jax.default_matmul_precision('highest'):
    O = jax.nn.dot_product_attention(Q, K, V, is_causal=True,
                                     local_window_size=(w - 1, 0))
    O_ref = windowed_attention(Q[0, :, 0], K[0, :, 0], V[0, :, 0], w)
print(f'maximum deviation: {float(jnp.abs(O[0, :, 0] - O_ref).max()):.2e}')

`jax.nn.dot_product_attention` takes the window as an argument:
`local_window_size=(w - 1, 0)` lets each query see $w - 1$ earlier keys and
none after it, and with `is_causal=True` this is the band of `window_mask`.
The output equals that of `windowed_attention`. XLA's default
implementation applies the window as a mask over the full score matrix;
`implementation='cudnn'` passes it to the fused kernel as a parameter
instead.

Fixed windows decide by position what may be attended, before the content is
seen. Natively sparse attention instead learns which blocks of keys each
query visits, at the same block granularity, and runs as a fused kernel
that visits only the chosen blocks [@Yuan.Gao.Dai.ea.2025].

## Linear Attention in Recurrent Form

### Kernelizing the Score

The third approach changes the softmax itself. Attention pooling with a
fixed kernel (that section) weights each value in
proportion to a similarity kernel, and the exponentiated dot product
$\exp(\mathbf{q}^\top\mathbf{k}/\sqrt{d})$ is one kernel among many. Suppose
we choose instead a kernel that *factorizes*,
$k(\mathbf{q}, \mathbf{k}) = \phi(\mathbf{q})^\top \phi(\mathbf{k})$ for
some elementwise feature map $\phi$. Causal attention with this kernel is

$$
\mathbf{o}_t^\top
= \frac{\sum_{s \leq t} \phi(\mathbf{q}_t)^\top \phi(\mathbf{k}_s)\,
        \mathbf{v}_s^\top}
       {\sum_{s \leq t} \phi(\mathbf{q}_t)^\top \phi(\mathbf{k}_s)}
= \frac{\phi(\mathbf{q}_t)^\top \mathbf{S}_t}
       {\phi(\mathbf{q}_t)^\top \mathbf{z}_t},
\qquad
\mathbf{S}_t = \sum_{s \leq t} \phi(\mathbf{k}_s) \mathbf{v}_s^\top,
\quad
\mathbf{z}_t = \sum_{s \leq t} \phi(\mathbf{k}_s).
$$

Because the query enters linearly, it factors out of the sum. The
remaining terms, $\mathbf{S}_t$ and $\mathbf{z}_t$, do not depend on the
query. Reordering the sums therefore removes the explicit pairwise
interaction and its quadratic cost. Instead of $t$ scores, each query needs
one matrix-vector product against a running summary
[@Katharopoulos.Vyas.Pappas.ea.2020]. Following that paper, we take
$\phi(x) = \mathrm{elu}(x) + 1$, which keeps features positive so that the
normalizer cannot vanish.

The sums in the equation satisfy the recurrences

$$
\mathbf{S}_t = \mathbf{S}_{t-1} + \phi(\mathbf{k}_t)\, \mathbf{v}_t^\top,
\qquad
\mathbf{z}_t = \mathbf{z}_{t-1} + \phi(\mathbf{k}_t),
$$

which define a fixed-size state of $d \times d_v$ numbers. The state is
updated by an outer-product write as each token arrives, then read by the
query. Linear attention is a recurrent network whose hidden state is a
matrix. The state acts as an associative memory, with $\phi(\mathbf{k}_t)$
the address and $\mathbf{v}_t$ the content.
@Schlag.Irie.Schmidhuber.2021 showed that this layer is the fast
weight programmer of the early 1990s. The same computation has two
implementations. The *parallel* form for training computes all states at
once with cumulative sums. The *recurrent* form for generation carries
$(\mathbf{S}, \mathbf{z})$ forward one token at a time, in constant memory,
like every RNN in that section.

In [ ]:
def phi(X):
    return jax.nn.elu(X) + 1

def linear_attention_parallel(Q, K, V):
    """Training form: all outputs at once via cumulative sums."""
    Qp, Kp = phi(Q), phi(K)
    S = jnp.cumsum(Kp[:, :, None] * V[:, None, :], axis=0)  # (n, d, d_v)
    z = jnp.cumsum(Kp, axis=0)                              # (n, d)
    return (Qp[:, :, None] * S).sum(1) / (Qp * z).sum(-1, keepdims=True)

def linear_attention_recurrent(Q, K, V):
    """Generation form: a (d, d_v) state, updated token by token."""
    Qp, Kp = phi(Q), phi(K)
    def step(carry, qkv):
        S, z = carry
        q, k, v = qkv
        S = S + k[:, None] * v[None, :]          # Outer-product write
        z = z + k
        return (S, z), q @ S / (q @ z)           # Query the memory
    init = (jnp.zeros((Q.shape[-1], V.shape[-1])), jnp.zeros(Q.shape[-1]))
    _, outputs = jax.lax.scan(step, init, (Qp, Kp, V))
    return outputs

Equations the equation and the equation
sum the same terms in different orders, so their implementations should
agree up to floating-point rounding.

In [ ]:
n = 512
Q, K, V = (jax.random.normal(k, (n, d_h))
           for k in jax.random.split(d2l.get_key(), 3))
err = jnp.abs(linear_attention_parallel(Q, K, V)
              - linear_attention_recurrent(Q, K, V)).max()
print(f'maximum deviation: {float(err):.2e}')
print(f'recurrent state: {d_h}x{d_h} + {d_h} floats '
      f'= {(d_h * d_h + d_h) * 4 / 1024:.0f} KiB at any sequence length')

At generation time this linear-attention layer with $d_h=64$ carries
16 KiB of state, whatever the context length. A softmax head instead caches
every past key and value, $2nd_h\cdot4$ bytes in fp32, or about 4 MiB at
$n=8192$. Its cache grows linearly with the generated sequence, whereas the
recurrent state stays fixed.

The state $\mathbf{S}_t$ is a lossy summary. It holds $d \times d_v$ numbers
however many tokens it has absorbed, so as the context grows the writes
interfere, and retrieving one particular value becomes approximate. Softmax
attention keeps every key and can put nearly all of a query's weight on one
of them. A kernel with positive features such as $\mathrm{elu} + 1$ cannot
become nearly one-hot in the same way: scaling up a query sharpens the
softmax without limit but leaves the normalized weights of this kernel almost
unchanged (Exercise 6).

### Chunked Computation

Neither form suits a GPU during training. The parallel form stores the state
at every position, $n$ matrices of $d \times d_v$ numbers: 256 MiB for one
head at $n = 16{,}384$ in fp32, more than the score matrix it replaces until
$n$ exceeds $d\, d_v = 4096$. The recurrent form stores one state but takes
$n$ sequential steps of matrix--vector work, which leaves the matrix units
of a GPU idle. Practical kernels divide the sequence into chunks of $c$
positions, as online softmax divided the keys [@Yang.Wang.Shen.ea.2024].
Inside a chunk, the outputs come from the quadratic form, a causally masked
$c \times c$ matrix of kernel values. Across chunks, the state carries the
sum over all earlier chunks. For a chunk that starts at position $t_0$, with
$\mathbf{S}$ and $\mathbf{z}$ the sums over all positions before $t_0$,

$$
\mathbf{o}_t^\top = \frac{\phi(\mathbf{q}_t)^\top \mathbf{S}
  + \sum_{t_0 \le s \le t} \phi(\mathbf{q}_t)^\top\phi(\mathbf{k}_s)\,
    \mathbf{v}_s^\top}
  {\phi(\mathbf{q}_t)^\top \mathbf{z}
  + \sum_{t_0 \le s \le t} \phi(\mathbf{q}_t)^\top\phi(\mathbf{k}_s)}.
$$

The first term of the numerator reads the memory of the earlier chunks, and
the second attends within the chunk. Both are matrix products. For each
chunk, a $c \times d$ by $d \times c$ product gives the kernel values, and
the state advances by a $d \times c$ by $c \times d_v$ product. The total
work is $O(ncd + nd\,d_v)$, linear in $n$, and only one state per
chunk is stored.

In [ ]:
def linear_attention_chunked(Q, K, V, c=64):
    """Training form in chunks: quadratic within, recurrent across."""
    n, d = Q.shape
    assert n % c == 0, 'chunks must tile the sequence exactly'
    Qp, Kp = phi(Q).reshape(n // c, c, d), phi(K).reshape(n // c, c, d)
    Vc = V.reshape(n // c, c, -1)
    # The state each chunk starts from: sums over all earlier chunks
    S = jnp.cumsum(Kp.swapaxes(-1, -2) @ Vc, axis=0)     # (n/c, d, d_v)
    z = jnp.cumsum(Kp.sum(1), axis=0)                    # (n/c, d)
    S = jnp.concatenate([jnp.zeros_like(S[:1]), S[:-1]])
    z = jnp.concatenate([jnp.zeros_like(z[:1]), z[:-1]])
    A = jnp.tril(Qp @ Kp.swapaxes(-1, -2))               # (n/c, c, c)
    num = Qp @ S + A @ Vc                                # Past + this chunk
    den = Qp @ z[..., None] + A.sum(-1, keepdims=True)
    return (num / den).reshape(n, -1)

The chunked form sums the same terms once more, so it too should match the
recurrent form.

In [ ]:
n = 512
Q, K, V = (jax.random.normal(k, (n, d_h))
           for k in jax.random.split(d2l.get_key(), 3))
with jax.default_matmul_precision('highest'):
    err = jnp.abs(linear_attention_chunked(Q, K, V)
                  - linear_attention_recurrent(Q, K, V)).max()
print(f'maximum deviation: {float(err):.2e}')

The three forms agree to floating-point rounding. They differ only in what
they keep in memory and in how much of the work is matrix products.

### Linear Attention in Practice

Plain kernelized attention of this kind did not replace softmax attention in
language models. At equal size it models language less well, and it is
weakest on tasks that must recall one particular earlier token
[@Arora.Eyuboglu.Zhang.ea.2024; @Yang.Wang.Shen.ea.2024]. Two changes
made the recurrent form competitive. First, the state learns to forget: a
decay computed from each input multiplies $\mathbf{S}_{t-1}$ before the new
write, so that old content fades when new content needs the room
(Exercise 10 works out the case
of a constant decay).
Second, models keep a few softmax layers for recall. MiniMax-01, for
instance, places one softmax attention layer after every seven
linear-attention layers [@MiniMax.2025]. A recurrence with a
fixed-size state that is updated linearly, as in
the equation, is a *linear state space model*.
that section develops these models and their input-dependent decay from
this starting point, and that section extends
`linear_attention_chunked` to a state that decays.

## Measured Running Time

We now time the three strategies of this section on equal terms, as the
sequence grows from 512 to 32,768 tokens: dense causal attention in the
fused kernel, windowed attention with $w = 128$, and chunked linear
attention. All three process 32 heads of width 64 at once, for instance a
minibatch of 4 sequences with 8 heads each. They run in bfloat16, the
precision in which attention is trained, and `vmap` maps the one-head
functions of this section over the heads. Enough work per call keeps the
fixed cost of launching kernels from hiding the growth with $n$. The
milliseconds depend on the GPU, but the slopes on the log--log plot follow
from the operation counts.

In [ ]:
H, lengths = 32, [512, 1024, 2048, 4096, 8192, 16384, 32768]
mechanisms = {
    'dense (fused)': jax.jit(
        lambda Q, K, V: fused_attention(Q[None], K[None], V[None])),
    'windowed (w=128)': jax.jit(jax.vmap(
        lambda Q, K, V: windowed_attention(Q, K, V, 128))),
    'linear (chunked)': jax.jit(jax.vmap(linear_attention_chunked))}
times = {name: [] for name in mechanisms}
for n in lengths:
    X = jax.random.normal(d2l.get_key(), (H, n, d_h), dtype=jnp.bfloat16)
    for name, f in mechanisms.items():
        times[name].append(d2l.Benchmark(lambda: f(X, X, X)).time * 1e3)
d2l.plot(lengths, list(times.values()), 'sequence length n', 'time (ms)',
         legend=list(times), xscale='log', yscale='log', figsize=(4.5, 3))

At short lengths all three take well under a millisecond, and fixed
per-call costs rather than the operation counts decide which is fastest. At
long lengths the time of dense attention grows about fourfold with every
doubling of $n$, while the times of the windowed and the chunked linear
mechanisms grow about twofold. At 32,768 tokens each of them therefore
needs well under half the time of dense attention, and the gap widens with
every further doubling. Dense attention remains the default at moderate
lengths, where it is exact and fast. At long contexts the mechanisms of
linear cost are faster, and linear attention is also faster during
generation, where its state stays the size of one matrix.

`jax.jit` compiles the windowed and the linear functions into a few fused
kernels, so their fixed cost is about that of the dense kernel. At short
lengths the three curves therefore lie close together, and the gap opens
only as $n$ grows.

## Summary

Dense attention costs $4n^2d$ FLOPs and, in a naive implementation, two
$n \times n$ buffers per head. Online softmax computes the same result one
key block at a time while carrying a running maximum, normalizer, and
output, and FlashAttention fits this schedule to the memory hierarchy, so
exact attention runs in linear memory and several times faster. Sliding
windows reduce the work to $O(nw)$ and widen the receptive field
to $1 + L(w-1)$ after $L$ layers; interleaved global layers restore direct
paths, and a fused kernel saves work only in whole tiles. A factorizing
kernel turns attention into a recurrent network with a $d \times d_v$ matrix
state, computed in chunks for training and step by step for generation; the
state has constant size but is a lossy summary of the past. Measured,
dense time grows fourfold per doubling of the length and the time of the
linear-cost mechanisms twofold, so at long contexts they need a fraction of
its time.

## Exercises

1. **Reaching across the sequence.** the table compares
   layers one at a time, but connecting two distant tokens may take many
   layers. Count one multiply--add as two FLOPs, so that a convolution
   layer with kernel $k$ costs $2knd^2$, an RNN layer $4nd^2$, and a
   self-attention layer $8nd^2 + 4n^2d$. Let $n = 4096$, $d = 512$, and
   $k = 3$.
    1. A stack of plain convolutions must connect the first token with the
       last. Find the number of layers it needs and its total FLOPs, show
       that this total grows quadratically in $n$, as the cost of one
       attention layer does, and compare the two totals numerically.
    1. Dilated convolutions (that section) with dilation
       $k^{l-1}$ in layer $l$ grow the receptive field geometrically. Find
       the receptive field after $L$ layers, the depth needed to connect
       the two tokens, and the total FLOPs of that stack.
    1. Rank the four designs, namely one attention layer, the two
       convolution stacks, and one RNN layer, by total FLOPs and by
       sequential steps, counting each layer of a stack as one step and
       each token of the RNN as one step. Which resource does each design
       spend to connect the two tokens?

1. [code] **The running maximum.** Remove the running maximum from
   `chunked_attention`, so that the state accumulates plain sums of
   $e^{a_j}$. Run the result with $\mathbf{Q}$, $\mathbf{K}$, $\mathbf{V}$
   and the state $s$, $\mathbf{O}$ in float16, on queries scaled up by a
   factor of $10$.
    1. Before running, compute the largest exponent $x$ for which $e^x$ is
       finite in float16, and estimate the fraction of scaled scores that
       exceed it when queries and keys of width $64$ have independent
       standard normal entries.
    1. Run the modified function, and report at which operation the first
       infinite or undefined value appears.
    1. Explain why carrying $m$ bounds every exponent by zero. A term
       $e^{a_j - m'}$ that is smaller than the smallest positive float16
       number rounds to zero. Bound the total contribution of such terms to
       $s$, relative to the term of the largest score, for a row of $n$
       keys.
    1. `chunked_attention` initializes $m$ with the most negative finite
       value of the dtype and masks scores with the same value, the idiom
       of that section. Take $n = 2c$ for
       blocks of $c$ keys, and let the loop visit the two key blocks in
       reverse order, so that every score of the first visited block is
       masked for a query in the first block. Trace $m$, $s$, and
       $\mathbf{O}$ for that query through both iterations, and show that
       the final output is still exact.

1. **Merging partial states.** Fix one query with scores $a_j$ and values
   $\mathbf{v}_j$. For a set $B$ of keys, call the triple $(m_B, s_B,
   \mathbf{o}_B)$ that the equation produces from the keys in
   $B$ alone the partial state of $B$, so that $\mathbf{o}_B / s_B$ is the
   attention output restricted to $B$.
    1. Derive the partial state of $B \cup B'$ for disjoint sets $B$ and
       $B'$ from the two partial states, and show that
       the equation is the special case in which $B'$ is a
       single block.
    1. Show that this merge is associative and commutative. Conclude that
       the key blocks can be processed in any order, or in parallel and
       merged in a tree of depth $\lceil \log_2 (n/c) \rceil$ for blocks of
       $c$ keys, and explain why this matters when a single query attends
       to a long cache during generation.
    1. Replace the pair $(m_B, s_B)$ by the single number $\ell_B = m_B +
       \log s_B$ and the triple by $(\ell_B, \bar{\mathbf{o}}_B)$ with
       $\bar{\mathbf{o}}_B = \mathbf{o}_B / s_B$. Express the merged
       $\ell$ and $\bar{\mathbf{o}}$ in these terms, and show that
       $\bar{\mathbf{o}}$ is a convex combination of
       $\bar{\mathbf{o}}_B$ and $\bar{\mathbf{o}}_{B'}$ whose weights
       depend only on $\ell_B$ and $\ell_{B'}$.

1. **Arithmetic intensity of attention.** The count
   the equation extends to values of width $d_v$: for $n$
   queries and keys of width $d$, the two matrix products of
   the equation cost $2n^2 d$ and $2n^2 d_v$ floating-point
   operations, and the softmax costs a few operations per score.
    1. Suppose that the score matrix is written to memory once and read
       once by the softmax, whose output is written once and read once by
       the value product, all in single precision. Count the bytes moved by
       these four passes, and by reading $\mathbf{Q}$, $\mathbf{K}$,
       $\mathbf{V}$ and writing the output.
    1. Neglecting the softmax arithmetic, form the ratio of operations to
       bytes and let $n \to \infty$ at fixed $d$ and $d_v$. Show that the
       limit is $(d + d_v)/8$, and state how the softmax arithmetic changes
       it.
    1. Look up the peak arithmetic throughput and the memory bandwidth of
       one accelerator, compute its ratio of operations per byte, and decide
       whether attention with $d = d_v = 64$ is limited by arithmetic or by
       memory traffic on it.

1. [code] **Tiles of a fused kernel.** A fused kernel works on tiles of
   $b \times b$ scores and skips a tile only if the mask excludes all of it.
   Let $T = n/b$.
    1. Show that a causal mask leaves $T(T+1)/2$ of the $T^2$ tiles, and
       that a window of width $w$, a multiple of $b$, leaves about
       $T(w/b + 1)$. Check both counts with `tiles_visited`. Below which
       width does narrowing the window stop saving tiles?
    1. Several documents are packed into one sequence of $n = 2048$ tokens,
       and the mask combines causality with "same document" as in
       that section, using `tiles_visited` and
       $b = 128$. Compare four documents of 512 tokens with documents of
       1,000, 600, 300, and 148 tokens. Explain why documents whose
       boundaries do not fall on tile boundaries cost more tiles.
    1. A kernel that receives the mask as an $n \times n$ boolean tensor
       must read it from memory. Compare the size of that tensor at
       $n = 32{,}768$ with the size of $\mathbf{Q}$, $\mathbf{K}$, and
       $\mathbf{V}$ for 8 heads of width 64 in bfloat16, and explain why
       FlexAttention evaluates the mask as a rule, once per tile.

1. [code] **The feature map.** The feature map $\phi(x) =
   \mathrm{elu}(x) + 1$ in `phi` is one choice among many. Compare it with
   $\phi(x) = \mathrm{relu}(x)$ and $\phi(x) = \exp(x)$, both applied
   elementwise, on $n = 512$ random queries and keys of width `d_h`.
    1. For each map, compute the causal linear-attention weight matrix
       with entries
       $$\frac{\phi(\mathbf{q}_t)^\top \phi(\mathbf{k}_s)}{\sum_{s' \leq t} \phi(\mathbf{q}_t)^\top \phi(\mathbf{k}_{s'})}, \qquad s \leq t,$$
       and measure its maximum absolute difference from the causal softmax
       weight matrix, which `causal_attention` forms before multiplying by
       $\mathbf{V}$, and the mean entropy of its rows.
    1. Under which condition on $\mathbf{q}_t$ and the keys
       $\mathbf{k}_1, \ldots, \mathbf{k}_t$ does the ReLU map produce an
       undefined row? Derive the probability of this event for row $t$
       when all entries are independent standard normal, and evaluate it
       for $t = 1$ at width $64$ and at width $4$.
    1. Multiply the queries by $\beta \in \{1, 4, 16\}$, and recompute the
       mean row entropy for the three maps and for the softmax weights.
       Explain from a property of each $\phi$ which maps cannot sharpen
       their weights in this way, and what this implies for approximating
       a softmax row that is nearly one-hot.

1. **Traffic of a tiled schedule.** ● By Exercise
   4a,
   the naive layer moves $4n^2$ floats to and from off-chip memory beyond
   the terms linear in $n$, since the score matrix and the weight matrix
   are each written once and read once. Ignore all traffic that is linear
   in $n$.
    1. Suppose that `chunked_attention` runs one operation at a time, each
       reading its inputs from off-chip memory and writing its output
       back. Count, as functions of $n$, $c$, and $d$, the floats moved for
       the score stripes, the floats moved for $\mathbf{Q}$ and the
       accumulator $\mathbf{O}$, and the number of operations launched.
       Which of these quantities does chunking reduce relative to the
       naive layer?
    1. A fused kernel instead holds a tile of $b$ queries, their output
       accumulator, and one tile each of $b$ keys and $b$ values in an
       on-chip memory of $M$ floats, so that $4bd \leq M$; scores and
       per-row statistics stay in registers. It loads each query tile once,
       streams all key and value tiles past it, and writes each output tile
       once. Show that its traffic beyond the linear terms is
       $2n^2 d / b$ floats and that the largest admissible tile gives
       $\Theta(n^2 d^2 / M)$, the bound that
       @Dao.Fu.Ermon.ea.2022 prove for FlashAttention.
    1. Compute the ratio of the fused traffic to the naive traffic for
       $d \in \{64, 128, 256\}$ with $M = 2^{16}$, and state how the head
       width limits the benefit of tiling.

1. **An exact feature map.** ● The softmax kernel $\kappa(\mathbf{q},
   \mathbf{k}) = \exp(\mathbf{q}^\top \mathbf{k} / \sqrt{d})$ also
   factorizes, but only with infinitely many features.
    1. Expand $\kappa$ in its Taylor series, and show that the degree-$p$
       term equals $\phi_p(\mathbf{q})^\top \phi_p(\mathbf{k})$ for a
       feature map $\phi_p$ of dimension $d^p$ built from the $p$-fold
       tensor product of its argument.
    1. Truncate the series after degree $2$. Show that the truncated kernel
       is positive for every pair of $\mathbf{q}$ and $\mathbf{k}$, so the
       normalizer of the equation cannot vanish, and give the
       dimension of the feature map that concatenates $\phi_0$, $\phi_1$,
       and $\phi_2$. For $d = 64$ and values of width $64$, compute the
       size of the state $(\mathbf{S}, \mathbf{z})$ in single precision and
       the length $n$ at which the key--value cache of a softmax head
       reaches the same size. A truncation of this kind is the feature map
       of Based [@Arora.Eyuboglu.Zhang.ea.2024].
    1. Take as given that the $n \times n$ matrix with entries
       $\kappa(\mathbf{x}_i, \mathbf{x}_j)$ is positive definite for
       distinct $\mathbf{x}_1, \ldots, \mathbf{x}_n$. Show that no feature
       map of dimension $D < n$ reproduces $\kappa$ exactly on these points
       when queries and keys coincide.

1. [extended] **Attention variants in a language model.** ● Whether the
   quality gap between exact and linear attention matters depends on the
   task. Measure it on `TinyCharLM` of
   that section, with `pos='learned'` for every
   variant so that position enters the same way.
    1. Subclass `TinyCharLM` and override `attend` twice: once with a
       causal band mask of width $w$ in place of the causal mask, and once
       with the causal linear-attention weights of the equation
       for $\phi(x) = \mathrm{elu}(x) + 1$, computed explicitly as a
       $T \times T$ matrix so that the method still returns its weights.
    1. Train dense attention, windows of width $w \in \{4, 16, 64\}$, and
       linear attention with the protocol of
       that section, and report validation perplexity at
       context $128$. Relate the window results to the reach $1 + 2(w-1)$
       of the two-block model.
    1. Build validation sequences of $128$ bytes, each a random
       $64$-byte excerpt of the validation text followed by the same
       excerpt again, and compare the mean loss on the second half with
       that on the first half for every variant. Which variants exploit the
       repetition, and how does the answer relate to the reach of the
       windowed models and to the fixed-size state of linear attention?

1. [code] **A decaying state.** ● Give the state of
   the equation a scalar decay $\gamma \in (0, 1)$,
   $$\mathbf{S}_t = \gamma\, \mathbf{S}_{t-1} + \phi(\mathbf{k}_t)\,\mathbf{v}_t^\top, \qquad \mathbf{z}_t = \gamma\, \mathbf{z}_{t-1} + \phi(\mathbf{k}_t).$$
    1. Show that the parallel form now weights the contribution of position
       $s$ to the output at $t$ by $\gamma^{t-s}$, and implement it by
       rescaling the cumulative sums of `linear_attention_parallel` with
       powers of $\gamma$.
    1. Add the decay to `linear_attention_recurrent`, and verify that the
       two forms agree for $n = 512$ and $\gamma = 0.99$.
    1. The rescaling multiplies position $s$ by $\gamma^{-s}$ before the
       cumulative sum. For which $n$ does this overflow single precision at
       $\gamma = 0.9$ and at $\gamma = 0.5$?
    1. Add the decay to `linear_attention_chunked`: rescale each position
       relative to the start of its chunk, and carry the state from one
       chunk to the next with the decayed recurrence. Show that no factor
       larger than $\gamma^{-c}$ appears, run the result for $\gamma = 0.5$,
       $n = 512$, and $c = 64$, and compare it with the recurrent form.
    1. Compute the lag at which the weight $\gamma^{t-s}$ falls below
       $0.01$ for $\gamma = 0.99$. A task requires recalling one token
       from $2000$ positions back, while the tokens in between are filler.
       Explain why no fixed $\gamma$ serves both that token and the
       filler, and what a decay computed from each token's own input could
       do instead.

[Discussions](https://d2l.discourse.group/t/36772)